In [ ]:
# ==== rumqtt post-quantum MQTT — single-cell Colab notebook ====
# Clones dhaneshkk/rumqtt and runs:
#   1) Lean4 proofs (pq-envelope/lean/HybridCombiner.lean): hybrid/X-Wing combiner,
#      KDF binding, MQTT 5 property check, bucket padding, replay window
#   2) rumqttd unit tests for the key_exchange / signatures (post-quantum login) policies
#   3) pq-envelope tests, each scored with MCC where it is a classifier:
#      HPKE (X-Wing) envelope, session mode with group keys + padding + zstd, key
#      management (certs, rotation, revocation), restarts, delayed delivery, and
#      end-to-end runs over rumqttd (MQTT 3.1.1 + 5, ML-DSA-65 mutual TLS logins)
# SAM is not applicable (no model is trained); there is no inference endpoint here.
import os, subprocess, shutil, pathlib

REPO = "https://github.com/dhaneshkk/rumqtt"
REF = "feat/pq-session-groups"   # switch to "main" after PR #3 is merged

HOME = pathlib.Path.home()
os.environ["PATH"] = f"{HOME}/.cargo/bin:{HOME}/.elan/bin:" + os.environ["PATH"]

def sh(cmd, cwd=None):
    print(f"\n$ {cmd}", flush=True)
    p = subprocess.Popen("set -o pipefail; " + cmd, shell=True, executable="/bin/bash",
                         cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise SystemExit(f"command failed ({p.returncode}): {cmd}")

if not shutil.which("cargo"):
    sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
if not shutil.which("cmake") or not shutil.which("clang"):
    sh("apt-get -qq update && apt-get -qq install -y cmake clang >/dev/null")
if not shutil.which("lean"):
    sh("curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | "
       "sh -s -- -y --default-toolchain leanprover/lean4:v4.24.0")

ROOT = pathlib.Path.cwd() / "rumqtt"
if not ROOT.exists():
    sh(f"git clone --depth 1 --branch {REF} {REPO} {ROOT}")

sh("lean --version && lean HybridCombiner.lean", cwd=ROOT / "pq-envelope" / "lean")
sh("cargo test -p rumqttd --lib tls 2>&1 | grep -E '^test |test result'", cwd=ROOT)
sh("cargo test -p rumqtt-pq-envelope -- --nocapture --test-threads=1", cwd=ROOT)
print("\nAll layers verified. Broker configs: rumqttd/rumqttd.toml (key_exchange), "
      "pq-envelope/broker/ (Mosquitto + OpenSSL).")
